# QLoRA training (Colab)

Run this on a Colab GPU runtime (Runtime > Change runtime type > GPU).

This notebook runs the **real 4-bit QLoRA** training path. It reuses the exact same
code as the repo (`train/qlora_train.py`) so the Colab run and the local CPU smoke
test can never drift: when CUDA is present, `qlora_train.py` loads the model in 4-bit
with Unsloth and trains with TRL's `SFTTrainer`; on CPU it falls back to plain `peft`
LoRA (that fallback is what proved the plumbing on Day 2).

Prereqs (Day 2, done): Behavior Spec written, data pipeline in `data/`, eval harness
in `eval/`, and the end-to-end smoke test passing. Day 3 uses this for the first real
training run on the v1 dataset.

## 1. Install environment

In [ ]:
!pip install -q unsloth trl datasets

## 2. Load base model and confirm GPU inference works

In [ ]:
from unsloth import FastLanguageModel

MODEL_ID = "Qwen/Qwen3-0.6B"  # placeholder base model, revisit once a behavior/size is chosen

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=2048,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

In [ ]:
messages = [{"role": "user", "content": "In one sentence, what is a QLoRA fine-tune?"}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

output = model.generate(**inputs, max_new_tokens=100)
print(tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

## 3. Train on the generated dataset (real QLoRA)

Clone the repo so the notebook runs the *same* code and dataset as everything else,
then invoke the training module. On a GPU runtime this takes the Unsloth 4-bit QLoRA
path automatically.

In [ ]:
# Get the repo (skip if you uploaded it instead). Replace with your remote.
%cd /content
![ -d SmallLearningModel] || git clone https://github.com/<your-org>/SmallLearningModel.git
%cd SmallLearningModel

# Real QLoRA run on the v1 dataset. On GPU this uses Unsloth 4-bit + TRL SFTTrainer.
# (For the Day-2 smoke dataset, swap --data data/generated_v0.jsonl and add --max-steps 10.)
!python -m train.qlora_train \
    --data data/generated_v1.jsonl \
    --adapter-name v1 \
    --epochs 3 \
    --lora-r 16 --lora-alpha 32 \
    --batch-size 8 --grad-accum 2 --lr 2e-4

## 4. Base-vs-tuned eval

Score the base model and the freshly trained adapter on the held-out concepts with the
identical minimal prompt, using the reused litmus FK scorer + accuracy judge. Set your
OpenAI key first (`import os; os.environ["OPENAI_API_KEY"]="..."`), or pass `--no-judge`
for an FK-only run.

In [ ]:
!python -m eval.base_vs_tuned --adapter train/adapters/v1 --limit 12